# Diabetic Patient Readmission Analysis Project

## Dataset Overview
- **Source**: Diabetic patient encounter data
- **Size**: 101,766 rows × 50 columns
- **Target**: Hospital readmission prediction (readmitted column)

## Architecture: Medallion (Bronze → Silver → Gold)

### 🥉 Bronze Layer
Raw data ingestion from CSV - no transformations
- Table: `diabetic_encounters_bronze`

### 🥈 Silver Layer
Cleaned and validated data
- Filter out unknown values (race = "?", medical_specialty = "?")
- Handle missing values
- Table: `diabetic_encounters_silver`

### 🥇 Gold Layer
Business-ready analytics tables and metric views
- Table: `diabetic_encounters` (production clean data)
- Metric View: `diabetic_patient_readmission_metrics` (powers the dashboard)

## Project Goals
1. Build a robust data pipeline (Bronze → Silver → Gold)
2. Analyze patient demographics, diagnoses, and medication patterns
3. Identify factors associated with hospital readmission
4. Build predictive models for readmission risk

## Setup Instructions
**Upload your `diabetic_data.csv` file** by dragging it into Cell 2, then run cells in order.

---
## 🥉 Bronze Layer: Raw Data Ingestion
Load CSV as-is with no transformations. This is the source of truth.

In [0]:
# 🥉 BRONZE: Create bronze table from existing data
# Using the existing diabetic_encounters table as the raw data source

from pyspark.sql import SparkSession

# Check if source table exists
source_table = "patient_readmission_analytics.default.diabetic_encounters"
print(f"Loading data from existing table: {source_table}")

try:
    df_bronze = spark.table(source_table)
    
    # Save as bronze table (raw source of truth)
    bronze_table = "patient_readmission_analytics.default.diabetic_encounters_bronze"
    df_bronze.write.mode("overwrite").saveAsTable(bronze_table)
    
    print(f"\n✓ Bronze table created: {bronze_table}")
    print(f"✓ Row count: {df_bronze.count():,}")
    print(f"✓ Columns: {len(df_bronze.columns)}")
    print("\n📝 Note: Using existing table as bronze source. This contains the raw uploaded data.")
except Exception as e:
    if "TABLE_OR_VIEW_NOT_FOUND" in str(e):
        print(f"❌ Source table '{source_table}' not found.")
        print("\n📌 Please upload diabetic_data.csv first:")
        print("1. Drag the CSV file into a cell above")
        print("2. Load it into the diabetic_encounters table")
    else:
        raise

---
## 🥈 Silver Layer: Data Cleaning & Validation
Apply data quality rules and filter out invalid/unknown values.

In [0]:
# 🥈 SILVER: Clean and validate data
from pyspark.sql.functions import col

# Load bronze table
bronze_table = "patient_readmission_analytics.default.diabetic_encounters_bronze"
df_bronze = spark.table(bronze_table)

print(f"Bronze table row count: {df_bronze.count():,}")

# Apply data quality filters
df_silver = df_bronze.filter(
    (col("race").isNotNull()) &
    (col("race") != "?") &
    (col("race") != "Unknown") &
    (col("medical_specialty").isNotNull()) &
    (col("medical_specialty") != "?")
)

# Save as silver table
silver_table = "patient_readmission_analytics.default.diabetic_encounters_silver"
df_silver.write.mode("overwrite").saveAsTable(silver_table)

silver_count = df_silver.count()
filtered_count = df_bronze.count() - silver_count

print(f"\n✓ Silver table created: {silver_table}")
print(f"✓ Clean records: {silver_count:,}")
print(f"✓ Filtered out: {filtered_count:,} records with unknown/invalid values")
print(f"✓ Data quality: {(silver_count / df_bronze.count() * 100):.1f}% retained")

---
## 🥇 Gold Layer: Production Analytics Tables
Create business-ready tables and metric views for dashboards and ML.

In [0]:
# 🥇 GOLD: Create production-ready table
# This is the table used by dashboards and ML models

silver_table = "patient_readmission_analytics.default.diabetic_encounters_silver"
df_silver = spark.table(silver_table)

# Create gold table (production clean data)
gold_table = "patient_readmission_analytics.default.diabetic_encounters"
df_silver.write.mode("overwrite").saveAsTable(gold_table)

print(f"✓ Gold table created: {gold_table}")
print(f"✓ Row count: {df_silver.count():,}")
print("\n✨ This is the production table used by:")
print("  - Dashboard metric views")
print("  - ML feature engineering")
print("  - Business analytics")

In [0]:
# 🥇 GOLD: Create Metric View for Dashboard
# This metric view powers the "Diabetic Patient Readmission Analysis" dashboard

view_sql = """
CREATE OR REPLACE VIEW patient_readmission_analytics.default.diabetic_patient_readmission_metrics AS
SELECT 
  age,
  gender,
  race,
  readmitted,
  diag_1,
  medical_specialty,
  time_in_hospital,
  num_medications,
  num_procedures,
  num_lab_procedures
FROM patient_readmission_analytics.default.diabetic_encounters
"""

spark.sql(view_sql)
print("✓ View created: patient_readmission_analytics.default.diabetic_patient_readmission_metrics")
print("✓ Dashboard can now query the clean gold data through this view!")
print("\n✨ Your pipeline is complete:")
print("   🥉 Bronze (101,766 raw rows)")
print("   🥈 Silver (50,727 clean rows)")
print("   🥇 Gold (50,727 production rows)")
print("   📊 View (dashboard-ready)")
print("\n🔗 All dashboard widgets will now show clean data without '?' values!")

In [0]:
# Load the GOLD table for analysis
table_name = "patient_readmission_analytics.default.diabetic_encounters"

# Check if table exists
try:
    df = spark.table(table_name)
    # Display basic information
    print(f"🥇 Gold Table: {table_name}")
    print(f"Total encounters: {df.count():,}")
    print(f"\nColumns ({len(df.columns)}):")
    df.printSchema()
except Exception as e:
    if "TABLE_OR_VIEW_NOT_FOUND" in str(e):
        print(f"❌ Table '{table_name}' does not exist yet.")
        print("\n📋 Please run the Bronze → Silver → Gold pipeline first (Cells 2-7).")
        print("\nSteps:")
        print("1. Upload diabetic_data.csv in Cell 2 (Bronze)")
        print("2. Run Cell 2 to create bronze table")
        print("3. Run Cell 4 to create silver table")
        print("4. Run Cells 6-7 to create gold table and metric view")
    else:
        raise

In [0]:
# Preview the data
display(df.limit(100))

---
## 🔍 Exploratory Data Analysis (Using Gold Data)
Analyze clean production data to understand readmission patterns.

In [0]:
# Analyze readmission rates
from pyspark.sql.functions import col, count, round as spark_round

readmission_stats = df.groupBy("readmitted").agg(
    count("*").alias("count")
).withColumn(
    "percentage",
    spark_round((col("count") / df.count() * 100), 2)
).orderBy("count", ascending=False)

display(readmission_stats)

In [0]:
# Analyze patient demographics
from pyspark.sql.functions import col, avg, countDistinct

# Age distribution by readmission status
age_analysis = df.groupBy("age", "readmitted").agg(
    count("*").alias("patient_count")
).orderBy("age", "readmitted")

display(age_analysis)

# Gender and race distribution
print("\n=== Gender Distribution ===")
display(df.groupBy("gender").count().orderBy("count", ascending=False))

print("\n=== Race Distribution ===")
display(df.groupBy("race").count().orderBy("count", ascending=False))

In [0]:
# Analyze length of hospital stay
from pyspark.sql.functions import avg, min, max, stddev

stay_stats = df.groupBy("readmitted").agg(
    avg("time_in_hospital").alias("avg_stay_days"),
    min("time_in_hospital").alias("min_stay_days"),
    max("time_in_hospital").alias("max_stay_days"),
    stddev("time_in_hospital").alias("stddev_stay_days")
)

display(stay_stats)

In [0]:
# Analyze number of medications and procedures
from pyspark.sql.functions import avg, col

medication_stats = df.groupBy("readmitted").agg(
    avg("num_medications").alias("avg_medications"),
    avg("num_procedures").alias("avg_procedures"),
    avg("num_lab_procedures").alias("avg_lab_procedures"),
    count("*").alias("patient_count")
)

display(medication_stats)

In [0]:
# Analyze primary diagnosis patterns
# ICD-9 diagnosis codes: diag_1, diag_2, diag_3

top_diagnoses = df.groupBy("diag_1").agg(
    count("*").alias("count")
).orderBy("count", ascending=False).limit(20)

print("Top 20 Primary Diagnoses (diag_1):")
display(top_diagnoses)

## Data Quality Check
Identify missing values and data quality issues.

In [0]:
# Check for missing values across all columns
from pyspark.sql.functions import col, sum as spark_sum, lit, when

total_rows = df.count()

# Calculate null counts for each column
missing_data = []
for column in df.columns:
    # Check if column is string type before checking for "?"
    col_type = dict(df.dtypes)[column]
    if col_type == "string":
        null_count = df.filter(col(column).isNull() | (col(column) == "?")).count()
    else:
        null_count = df.filter(col(column).isNull()).count()
    
    if null_count > 0:
        missing_data.append({
            "column": column,
            "null_count": null_count,
            "null_percentage": round(null_count / total_rows * 100, 2)
        })

# Convert to DataFrame and display
if missing_data:
    missing_df = spark.createDataFrame(missing_data)
    display(missing_df.orderBy("null_percentage", ascending=False))
else:
    print("No missing values found!")